# LIBRARIES

In [54]:
## Autorreload all the files
%load_ext autoreload
%autoreload 2


import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.io.arff import loadarff
import pandas as pd
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.metrics import mean_squared_error, accuracy_score#, root_mean_squared_error
## Import every file in ../Scripts/
import os
import sys
sys.path.append('../Scripts')
sys.path.append('MonoKAN/Scripts/')
sys.path.append('MonoKAN/loaders/')
import KANLayer
import KAN
import spline
import utils
import importlib
import CustomELU
import grid_search
importlib.reload(KANLayer)
importlib.reload(KAN)
importlib.reload(spline)
importlib.reload(utils)
importlib.reload(CustomELU)
importlib.reload(grid_search)

from KANLayer import KANLayer
from KAN import KAN
from CustomELU import CustomELU
from grid_search import grid_search

import torch.nn as nn
import torch
import torch.nn as nn

#from mono_dense_keras.experiments import (
#    create_tuner_stats,
#    find_hyperparameters,
#    get_train_n_test_data,
#)
#torch.set_printoptions(sci_mode=False, precision=3)
print("Device", torch.cuda.is_available())

## Ignore future warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Device True


# AUTOMPG

In [55]:
torch.manual_seed(0)
torch.use_deterministic_algorithms(True)
neurons_list = [[2,3,1]]  # example neuron configurations
k_list = [-1]
lambda_l1_list = [0.0]
lambda_entropy_list = [1.0]
seeds = [0, 1, 2]

# Define other fixed parameters
grid = [10]
noise_scale = 0.0
noise_scale_base = 0.0
grid_eps = 0
symbolic_enabled = False
opt = "Adam"
steps = 5000
lamb = 1.00
small_reg_factor = 0
update_grid = False
lr_list = [0.01]
monotonic = True
base_function = torch.nn.Sigmoid()
patience = 100
batch = -1
hermite = True
classification = False
mono_vars = {0:0,1:-1,2:-1,3:-1,4:0,5:0,6:0}


In [3]:
grid_search(dataset_name='auto',neurons_list=neurons_list, k_list=k_list, lambda_l1_list=lambda_l1_list,
            lambda_entropy_list=lambda_entropy_list, seeds=seeds
                ,grids=grid, noise_scale=noise_scale, noise_scale_base=noise_scale_base, grid_eps=grid_eps,
             symbolic_enabled=symbolic_enabled, opt=opt, patience=patience, hermite=hermite,normalize=False,
                steps=steps, lamb=lamb, small_reg_factor=small_reg_factor,
             update_grid=update_grid, lr_list=lr_list, monotonic=monotonic, base_function=base_function,
            batch_size=batch, device = 'cpu')

Number of instances in train_data: 250
Number of instances in test_data: 79
Number of instances in val_data: 63
Training model with: seed=0, neurons=[7, 2, 3, 1], k=-1, grid=10, lambda_l1=0.0, lambda_entropy=1.0, lr=0.01


Epoch: 811/5000 | train loss: 4.82e+00 | val loss: 8.87e+00 | best val loss: 8.11e+00 | reg: 7.88e+0


Early stopping
The model has 558 parameters.
Training time: 7.12 seconds
Model device: cpu
Model saved to ../models/auto/model_seed_0.pt
Training model with: seed=1, neurons=[7, 2, 3, 1], k=-1, grid=10, lambda_l1=0.0, lambda_entropy=1.0, lr=0.01


Epoch: 811/5000 | train loss: 4.81e+00 | val loss: 8.80e+00 | best val loss: 8.20e+00 | reg: 7.87e+0


Early stopping
The model has 558 parameters.
Training time: 7.93 seconds
Model device: cpu
Model saved to ../models/auto/model_seed_1.pt
Training model with: seed=2, neurons=[7, 2, 3, 1], k=-1, grid=10, lambda_l1=0.0, lambda_entropy=1.0, lr=0.01


Epoch: 811/5000 | train loss: 4.80e+00 | val loss: 8.93e+00 | best val loss: 8.21e+00 | reg: 7.88e+0


Early stopping
The model has 558 parameters.
Training time: 8.09 seconds
Model device: cpu
Model saved to ../models/auto/model_seed_2.pt
Grid search completed and results saved to ./exp_results/auto_grid_search_results.csv


In [57]:
results = pd.read_csv('./exp_results/auto_grid_search_results.csv')
## Compute the mean
mean_values = results['test'].mean().round(2)
print(f'Mean of the values: {mean_values}')
## Compute the standard deviation
std_dev = results['test'].std()
print(f'Standard deviation: {std_dev}')
# Compute the mean training time
print(f'Mean training time: {results["training_time"].mean().round(2)} seconds')
# Compute the standard deviation time
print(f'Standard deviation time: {results["training_time"].std()} seconds')

results.head()

Mean of the values: 6.18
Standard deviation: 0.027715772652058216
Mean training time: 7.24 seconds
Standard deviation time: 0.1970032812334326 seconds


,neurons,k,lamb,noise_scale,noise_scale_base,lambda_l1,lambda_entropy,grid,seed,opt,patience,hermite,normalize,lr,batch_size,train,val,test,training_time,number_of_parameters
0,"[7, 2, 3, 1]",-1,1.0,0.0,0.0,0.0,1.0,10,0,Adam,100,True,False,0.01,-1,5.655002,8.111500,6.212656,7.019532,558
1,"[7, 2, 3, 1]",-1,1.0,0.0,0.0,0.0,1.0,10,1,Adam,100,True,False,0.01,-1,5.599606,8.204187,6.177422,7.310486,558
2,"[7, 2, 3, 1]",-1,1.0,0.0,0.0,0.0,1.0,10,2,Adam,100,True,False,0.01,-1,5.617386,8.206107,6.157979,7.395097,558


# HEART DISEASE

In [58]:
torch.manual_seed(0)
torch.use_deterministic_algorithms(True)

neurons_list = [[5,3,1]]  # example neuron configurations
k_list = [-1]
lambda_l1_list = [0.0]
lambda_entropy_list = [0.0]
seeds = [0, 1, 2]

# Define other fixed parameters
grids = [7]
noise_scale = 0.1
noise_scale_base = 0.1
grid_eps = 0
symbolic_enabled = False
opt = "Adam"
steps = 5000
lamb = 0.00
small_reg_factor = 0
update_grid = False
lr_list = [0.05]
monotonic = True
base_function = torch.nn.Sigmoid()
patience = 100
batch = -1
hermite = True
classification = True


In [59]:
dataset_name ='heart'
grid_search(dataset_name='heart',neurons_list=neurons_list, k_list=k_list, lambda_l1_list=lambda_l1_list,
            lambda_entropy_list=lambda_entropy_list, seeds=seeds
                ,grids=grids, noise_scale=noise_scale, noise_scale_base=noise_scale_base, grid_eps=grid_eps,
             symbolic_enabled=symbolic_enabled, opt=opt, patience=patience, hermite=hermite,normalize=False,
                steps=steps, lamb=lamb, small_reg_factor=small_reg_factor,
             update_grid=update_grid, lr_list=lr_list, monotonic=monotonic, base_function=base_function,batch_size=batch, device='cpu')

Number of instances in train_data: torch.Size([193, 13])
Number of instances in val_data: torch.Size([49, 13])
Number of instances in test_data: torch.Size([61, 13])
Number of instances in train_data: 193
Number of instances in test_data: 61
Number of instances in val_data: 49
Training model with: seed=0, neurons=[13, 5, 3, 1], k=-1, grid=7, lambda_l1=0.0, lambda_entropy=0.0, lr=0.05


Epoch: 134/5000 | train loss: 1.47e-01 | val loss: 9.78e-01 | best val loss: 4.77e-01 | reg: 0.00e+0


Early stopping
The model has 1503 parameters.
Training time: 1.50 seconds
Model device: cpu
Train Accuracy 0.8756476683937824
Val Accuracy 0.8571428571428571
Test Accuracy 0.8852459016393442
Model saved to ../models/heart/model_seed_0.pt
Training model with: seed=1, neurons=[13, 5, 3, 1], k=-1, grid=7, lambda_l1=0.0, lambda_entropy=0.0, lr=0.05


Epoch: 141/5000 | train loss: 1.47e-01 | val loss: 4.11e+00 | best val loss: 5.35e-01 | reg: 0.00e+0


Early stopping
The model has 1503 parameters.
Training time: 1.57 seconds
Model device: cpu
Train Accuracy 0.9067357512953368
Val Accuracy 0.8571428571428571
Test Accuracy 0.8852459016393442
Model saved to ../models/heart/model_seed_1.pt
Training model with: seed=2, neurons=[13, 5, 3, 1], k=-1, grid=7, lambda_l1=0.0, lambda_entropy=0.0, lr=0.05


Epoch: 138/5000 | train loss: 1.66e-01 | val loss: 4.52e+00 | best val loss: 5.37e-01 | reg: 0.00e+0


Early stopping
The model has 1503 parameters.
Training time: 1.51 seconds
Model device: cpu
Train Accuracy 0.8860103626943006
Val Accuracy 0.8571428571428571
Test Accuracy 0.8852459016393442
Model saved to ../models/heart/model_seed_2.pt
Grid search completed and results saved to ./exp_results/heart_grid_search_results.csv


In [4]:
dataset_name ='heart'
grid_search(dataset_name='heart',neurons_list=neurons_list, k_list=k_list, lambda_l1_list=lambda_l1_list,
            lambda_entropy_list=lambda_entropy_list, seeds=seeds
                ,grids=grids, noise_scale=noise_scale, noise_scale_base=noise_scale_base, grid_eps=grid_eps,
             symbolic_enabled=symbolic_enabled, opt=opt, patience=patience, hermite=hermite,normalize=False,
                steps=steps, lamb=lamb, small_reg_factor=small_reg_factor,
             update_grid=update_grid, lr_list=lr_list, monotonic=monotonic, base_function=base_function,batch_size=batch, device='cpu')

Number of instances in train_data: torch.Size([193, 13])
Number of instances in val_data: torch.Size([49, 13])
Number of instances in test_data: torch.Size([61, 13])
Number of instances in train_data: 193
Number of instances in test_data: 61
Number of instances in val_data: 49
Training model with: seed=0, neurons=[13, 5, 3, 1], k=-1, grid=7, lambda_l1=0.0, lambda_entropy=0.0, lr=0.05


Epoch: 134/5000 | train loss: 1.47e-01 | val loss: 9.78e-01 | best val loss: 4.77e-01 | reg: 0.00e+0


Early stopping
The model has 1503 parameters.
Training time: 2.15 seconds
Model device: cpu
Train Accuracy 0.8756476683937824
Val Accuracy 0.8571428571428571
Test Accuracy 0.8852459016393442
Model saved to ../models/heart/model_seed_0.pt
Training model with: seed=1, neurons=[13, 5, 3, 1], k=-1, grid=7, lambda_l1=0.0, lambda_entropy=0.0, lr=0.05


Epoch: 141/5000 | train loss: 1.47e-01 | val loss: 4.11e+00 | best val loss: 5.35e-01 | reg: 0.00e+0


Early stopping
The model has 1503 parameters.
Training time: 1.71 seconds
Model device: cpu
Train Accuracy 0.9067357512953368
Val Accuracy 0.8571428571428571
Test Accuracy 0.8852459016393442
Model saved to ../models/heart/model_seed_1.pt
Training model with: seed=2, neurons=[13, 5, 3, 1], k=-1, grid=7, lambda_l1=0.0, lambda_entropy=0.0, lr=0.05


Epoch: 138/5000 | train loss: 1.66e-01 | val loss: 4.52e+00 | best val loss: 5.37e-01 | reg: 0.00e+0


Early stopping
The model has 1503 parameters.
Training time: 1.69 seconds
Model device: cpu
Train Accuracy 0.8860103626943006
Val Accuracy 0.8571428571428571
Test Accuracy 0.8852459016393442
Model saved to ../models/heart/model_seed_2.pt
Grid search completed and results saved to ./exp_results/heart_grid_search_results.csv


In [5]:
results = pd.read_csv('./exp_results/heart_grid_search_results.csv')
## Compute the mean
mean_values = results['test'].mean().round(2)
print(f'Mean of the values: {mean_values}')
## Compute the standard deviation
std_dev = results['test'].std().round(2)
print(f'Standard deviation: {std_dev}')
print(f'Standard deviation: {std_dev}')
# Compute the mean training time
print(f'Mean training time: {results["training_time"].mean().round(2)} seconds')
# Compute the standard deviation time
print(f'Standard deviation time: {results["training_time"].std().round(2)} seconds')

results.head()

Mean of the values: 0.89
Standard deviation: 0.0
Standard deviation: 0.0
Mean training time: 1.85 seconds
Standard deviation time: 0.26 seconds


,neurons,k,lamb,noise_scale,noise_scale_base,lambda_l1,lambda_entropy,grid,seed,opt,patience,hermite,normalize,lr,batch_size,train,val,test,training_time,number_of_parameters
0,"[13, 5, 3, 1]",-1,0.0,0.1,0.1,0.0,0.0,7,0,Adam,100,True,False,0.05,-1,0.875648,0.857143,0.885246,2.151426,1503
1,"[13, 5, 3, 1]",-1,0.0,0.1,0.1,0.0,0.0,7,1,Adam,100,True,False,0.05,-1,0.906736,0.857143,0.885246,1.714806,1503
2,"[13, 5, 3, 1]",-1,0.0,0.1,0.1,0.0,0.0,7,2,Adam,100,True,False,0.05,-1,0.886010,0.857143,0.885246,1.691400,1503


# COMPAS

In [14]:

torch.manual_seed(0)
torch.use_deterministic_algorithms(True)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device = 'cpu'

neurons_list = [[5,10,5,1]]  # example neuron configurations
k_list = [-1]
lambda_l1_list = [0.0]
lambda_entropy_list = [0.0]
seeds = [0, 1, 2]

# Define other fixed parameters
grids = [10]
noise_scale = 0.1
noise_scale_base = 0.1
grid_eps = 0
symbolic_enabled = False
opt = "Adam"
steps = 5000
lamb = 0.00
small_reg_factor = 0
update_grid = False
lr_list = [0.001]
monotonic = True
base_function = torch.nn.Sigmoid()
patience = 500
batch = -1
hermite = True
classification = True
# monotone_constraints = [1 if i in mono_list else 0 for i in range(X_train.shape[1])]
# mono_vars = {i: value for i, value in enumerate(monotone_constraints)}

In [15]:
grid_search(dataset_name='compas',neurons_list=neurons_list, k_list=k_list, lambda_l1_list=lambda_l1_list,
            lambda_entropy_list=lambda_entropy_list, seeds=seeds
                ,grids=grids, noise_scale=noise_scale, noise_scale_base=noise_scale_base, grid_eps=grid_eps,
             symbolic_enabled=symbolic_enabled, opt=opt, patience=patience, hermite=hermite,normalize=False,
                steps=steps, lamb=lamb, small_reg_factor=small_reg_factor,
             update_grid=update_grid, lr_list=lr_list, monotonic=monotonic, base_function=base_function,batch_size=batch, device = device)

Number of instances in train_data: torch.Size([3949, 13])
Number of instances in val_data: torch.Size([988, 13])
Number of instances in test_data: torch.Size([1235, 13])
Number of instances in train_data: 3949
Number of instances in test_data: 1235
Number of instances in val_data: 988
Training model with: seed=0, neurons=[13, 5, 10, 5, 1], k=-1, grid=10, lambda_l1=0.0, lambda_entropy=0.0, lr=0.001


Epoch: 846/5000 | train loss: 5.87e-01 | val loss: 6.15e-01 | best val loss: 6.13e-01 | reg: 0.00e+0


Early stopping
The model has 4101 parameters.
Training time: 44.55 seconds
Model device: cpu
Train Accuracy 0.6816915674854394
Val Accuracy 0.6882591093117408
Test Accuracy 0.6898785425101215
Model saved to ../models/compas/model_seed_0.pt
Training model with: seed=1, neurons=[13, 5, 10, 5, 1], k=-1, grid=10, lambda_l1=0.0, lambda_entropy=0.0, lr=0.001


Epoch: 931/5000 | train loss: 5.90e-01 | val loss: 6.12e-01 | best val loss: 6.12e-01 | reg: 0.00e+0


Early stopping
The model has 4101 parameters.
Training time: 52.32 seconds
Model device: cpu
Train Accuracy 0.6816915674854394
Val Accuracy 0.6902834008097166
Test Accuracy 0.6890688259109312
Model saved to ../models/compas/model_seed_1.pt
Training model with: seed=2, neurons=[13, 5, 10, 5, 1], k=-1, grid=10, lambda_l1=0.0, lambda_entropy=0.0, lr=0.001


Epoch: 892/5000 | train loss: 5.88e-01 | val loss: 6.15e-01 | best val loss: 6.14e-01 | reg: 0.00e+0


Early stopping
The model has 4101 parameters.
Training time: 50.01 seconds
Model device: cpu
Train Accuracy 0.6827044821473791
Val Accuracy 0.687246963562753
Test Accuracy 0.6898785425101215
Model saved to ../models/compas/model_seed_2.pt
Grid search completed and results saved to ./exp_results/compas_grid_search_results.csv


In [16]:
results = pd.read_csv('./exp_results/compas_grid_search_results.csv')
## Compute the mean of the first 3 values
mean_first_3 = results['test'].iloc[:3].mean().round(3)*100
print(f'Mean of the first 3 values: {mean_first_3}')
## Compute the standard deviation
std_dev = results['test'].iloc[:3].std()*100
print(f'Standard deviation: {std_dev}')
# Compute the mean training time
print(f'Mean training time: {results["training_time"].mean().round(2)} seconds')
# Compute the standard deviation time
print(f'Standard deviation time: {results["training_time"].std()} seconds')

results.head()

Mean of the first 3 values: 69.0
Standard deviation: 0.046749009650981
Mean training time: 48.96 seconds
Standard deviation time: 3.993002364054981 seconds


,neurons,k,lamb,noise_scale,noise_scale_base,lambda_l1,lambda_entropy,grid,seed,opt,patience,hermite,normalize,lr,batch_size,train,val,test,training_time,number_of_parameters
0,"[13, 5, 10, 5, 1]",-1,0.0,0.1,0.1,0.0,0.0,10,0,Adam,500,True,False,0.001,-1,0.681692,0.688259,0.689879,44.548566,4101
1,"[13, 5, 10, 5, 1]",-1,0.0,0.1,0.1,0.0,0.0,10,1,Adam,500,True,False,0.001,-1,0.681692,0.690283,0.689069,52.324587,4101
2,"[13, 5, 10, 5, 1]",-1,0.0,0.1,0.1,0.0,0.0,10,2,Adam,500,True,False,0.001,-1,0.682704,0.687247,0.689879,50.012115,4101


# LOAN

In [65]:
torch.manual_seed(0)
torch.use_deterministic_algorithms(True)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# neurons_list = [[10,5,1]]
# neurons_list = [[10,10,5,1]]
neurons_list = [[5,1]]

k_list = [-1]
lambda_l1_list = [0.0]
lambda_entropy_list = [0.0]
seeds = [0, 1, 2]

# Define other fixed parameters
grids = [10]
noise_scale = 0.1
noise_scale_base = 0.1
grid_eps = 0
symbolic_enabled = False
opt = "Adam"
steps = 5000
lamb = 0.00
small_reg_factor = 0
update_grid = False
lr_list = [0.01]
monotonic = True
base_function = torch.nn.Sigmoid()
patience = 1000
batch = 334957//10
hermite = True
classification = True


# #model = KAN(width=[n_var,5,10,1], grid=10,grid_range=[ini,fin], k=-1,
# #            noise_scale=0.1,noise_scale_base=0.1, seed=seed,grid_eps=0,
# #            base_fun=nn.Sigmoid(),symbolic_enabled=False,classification=classification,hermite=hermite)

# monotone_constraints = np.array([int(i in mono_list) for i in range(original_X_train.shape[1])])
# if ridged:
#     monotone_constraints = monotone_constraints[top_features]
# mono_vars = {i: value for i, value in enumerate(monotone_constraints)}


In [24]:
grid_search(dataset_name='loan',neurons_list=neurons_list, k_list=k_list, lambda_l1_list=lambda_l1_list,
            lambda_entropy_list=lambda_entropy_list, seeds=seeds
                ,grids=grids, noise_scale=noise_scale, noise_scale_base=noise_scale_base, grid_eps=grid_eps,
             symbolic_enabled=symbolic_enabled, opt=opt, patience=patience, hermite=hermite,normalize=False,
                steps=steps, lamb=lamb, small_reg_factor=small_reg_factor,
             update_grid=update_grid, lr_list=lr_list, monotonic=monotonic, base_function=base_function,batch_size=batch, device = device)

Ridge Regression
Number of instances in train_data: torch.Size([334957, 15])
Number of instances in val_data: torch.Size([83740, 15])
Number of instances in test_data: torch.Size([70212, 15])
Number of instances in train_data: 334957
Number of instances in test_data: 70212
Number of instances in val_data: 83740
Training model with: seed=0, neurons=[15, 5, 1], k=-1, grid=10, lambda_l1=0.0, lambda_entropy=0.0, lr=0.01


Epoch: 2167/5000 | train loss: 6.22e-01 | val loss: 6.24e-01 | best val loss: 6.24e-01 | reg: 0.00e+


Early stopping
The model has 1926 parameters.
Training time: 138.63 seconds
Model device: cuda:0
Train Accuracy 0.6490534605934493
Val Accuracy 0.6489371865297349
Test Accuracy 0.6537344043753205
Model saved to ../models/loan/model_seed_0.pt
Training model with: seed=1, neurons=[15, 5, 1], k=-1, grid=10, lambda_l1=0.0, lambda_entropy=0.0, lr=0.01


Epoch: 2604/5000 | train loss: 6.24e-01 | val loss: 6.24e-01 | best val loss: 6.24e-01 | reg: 0.00e+


Early stopping
The model has 1926 parameters.
Training time: 162.84 seconds
Model device: cuda:0
Train Accuracy 0.6489489695692283
Val Accuracy 0.6490088368760449
Test Accuracy 0.6526092405856548
Model saved to ../models/loan/model_seed_1.pt
Training model with: seed=2, neurons=[15, 5, 1], k=-1, grid=10, lambda_l1=0.0, lambda_entropy=0.0, lr=0.01


Epoch: 2313/5000 | train loss: 6.22e-01 | val loss: 6.25e-01 | best val loss: 6.24e-01 | reg: 0.00e+

Early stopping
The model has 1926 parameters.
Training time: 146.52 seconds
Model device: cuda:0
Train Accuracy 0.6488564203763468
Val Accuracy 0.6481251492715548
Test Accuracy 0.652694696063351
Model saved to ../models/loan/model_seed_2.pt
Grid search completed and results saved to ./exp_results/loan_grid_search_results.csv


In [26]:
results = pd.read_csv('./exp_results/loan_grid_search_results.csv')
## Compute the mean of the first 3 values
mean_first_3 = results['test'].iloc[:3].mean().round(3)*100
print(f'Mean of the first 3 values: {mean_first_3}')
## Compute the standard deviation
std_dev = results['test'].iloc[:3].std()*100
print(f'Standard deviation: {std_dev}')
# Compute the mean training time
print(f'Mean training time: {results["training_time"].mean().round(2)} seconds')
# Compute the standard deviation time
print(f'Standard deviation time: {results["training_time"].std()} seconds')

results.head()

Mean of the first 3 values: 65.3
Standard deviation: 0.06264036991990572
Mean training time: 149.33 seconds
Standard deviation time: 12.343932512871945 seconds


,neurons,k,lamb,noise_scale,noise_scale_base,lambda_l1,lambda_entropy,grid,seed,opt,patience,hermite,normalize,lr,batch_size,train,val,test,training_time,number_of_parameters
0,"[15, 5, 1]",-1,0.0,0.1,0.1,0.0,0.0,10,0,Adam,1000,True,False,0.01,33495,0.649053,0.648937,0.653734,138.633677,1926
1,"[15, 5, 1]",-1,0.0,0.1,0.1,0.0,0.0,10,1,Adam,1000,True,False,0.01,33495,0.648949,0.649009,0.652609,162.837143,1926
2,"[15, 5, 1]",-1,0.0,0.1,0.1,0.0,0.0,10,2,Adam,1000,True,False,0.01,33495,0.648856,0.648125,0.652695,146.520888,1926


# BLOG

In [61]:
torch.manual_seed(0)
torch.use_deterministic_algorithms(True)
# neurons_list = [[10,5,1]]
neurons_list = [[10,10,5,1]]
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

k_list = [-1]
lambda_l1_list = [0.0]
lambda_entropy_list = [0.0]
seeds = [0,1,2]

# model = KAN(width=[n_var,5,10,5,1], grid=10,grid_range=[ini,fin], k=-1,
#             noise_scale=0.1,noise_scale_base=0.1, seed=seed,grid_eps=0,
#             base_fun=nn.Sigmoid(),symbolic_enabled=False,classification=classification,hermite=hermite)
# Define other fixed parameters
grids = [10]
noise_scale = 0.1
noise_scale_base = 0.1
grid_eps = 0
symbolic_enabled = False
opt = "Adam"
steps = 7000
lamb = 0.00
small_reg_factor = 0
update_grid = False
lr_list = [0.01]
monotonic = True
base_function = torch.nn.Sigmoid()
patience = 2000
batch = 418697
# batch = 418697//100
hermite = True
classification = False



In [51]:
grid_search(dataset_name='blog',neurons_list=neurons_list, k_list=k_list, lambda_l1_list=lambda_l1_list,
            lambda_entropy_list=lambda_entropy_list, seeds=seeds
                ,grids=grids, noise_scale=noise_scale, noise_scale_base=noise_scale_base, grid_eps=grid_eps,
             symbolic_enabled=symbolic_enabled, opt=opt, patience=patience, hermite=hermite,normalize=False,
                steps=steps, lamb=lamb, small_reg_factor=small_reg_factor,
             update_grid=update_grid, lr_list=lr_list, monotonic=monotonic, base_function=base_function,batch_size=batch, device = device)

Ridge Regression
Number of instances in train_data: torch.Size([37841, 20])
Number of instances in val_data: torch.Size([9461, 20])
Number of instances in test_data: torch.Size([6968, 20])
Number of instances in train_data: 37841
Number of instances in test_data: 6968
Number of instances in val_data: 9461
Training model with: seed=0, neurons=[20, 10, 10, 5, 1], k=-1, grid=10, lambda_l1=0.0, lambda_entropy=0.0, lr=0.01


Epoch: 3296/7000 | train loss: 2.71e-02 | val loss: 2.79e-02 | best val loss: 2.65e-02 | reg: 0.00e+


Early stopping
The model has 8546 parameters.
Training time: 345.83 seconds
Model device: cuda:0
Model saved to ../models/blog/model_seed_0.pt
Training model with: seed=1, neurons=[20, 10, 10, 5, 1], k=-1, grid=10, lambda_l1=0.0, lambda_entropy=0.0, lr=0.01


Epoch: 5163/7000 | train loss: 2.67e-02 | val loss: 2.92e-02 | best val loss: 2.64e-02 | reg: 0.00e+


Early stopping
The model has 8546 parameters.
Training time: 534.30 seconds
Model device: cuda:0
Model saved to ../models/blog/model_seed_1.pt
Training model with: seed=2, neurons=[20, 10, 10, 5, 1], k=-1, grid=10, lambda_l1=0.0, lambda_entropy=0.0, lr=0.01


Epoch: 6999/7000 | train loss: 2.55e-02 | val loss: 2.57e-02 | best val loss: 2.56e-02 | reg: 0.00e+


The model has 8546 parameters.
Training time: 699.51 seconds
Model device: cuda:0
Model saved to ../models/blog/model_seed_2.pt
Grid search completed and results saved to ./exp_results/blog_grid_search_results.csv


In [53]:
import math
results = pd.read_csv('./exp_results/blog_grid_search_results.csv')
## Compute the mean
# Take square root of 'train' and 'test' columns if they are MSE
results['train'] = results['train'].apply(lambda x: math.sqrt(x))
results['test'] = results['test'].apply(lambda x: math.sqrt(x))

mean_values = results['test'].mean().round(3)
print(f'Mean of the values: {mean_values}')
## Compute the standard deviation
std_dev = results['test'].std()
print(f'Standard deviation: {std_dev}')
# Compute the mean training time
print(f'Mean training time: {results["training_time"].mean().round(2)} seconds')
# Compute the standard deviation time
print(f'Standard deviation time: {results["training_time"].std()} seconds')

results.head()

Mean of the values: 0.155
Standard deviation: 0.00031916363723194914
Mean training time: 526.55 seconds
Standard deviation time: 176.96601122978458 seconds


,neurons,k,lamb,noise_scale,noise_scale_base,lambda_l1,lambda_entropy,grid,seed,opt,patience,hermite,normalize,lr,batch_size,train,val,test,training_time,number_of_parameters
0,"[20, 10, 10, 5, 1]",-1,0.0,0.1,0.1,0.0,0.0,10,0,Adam,2000,True,False,0.01,418697,0.165321,0.026472,0.155322,345.831793,8546
1,"[20, 10, 10, 5, 1]",-1,0.0,0.1,0.1,0.0,0.0,10,1,Adam,2000,True,False,0.01,418697,0.164157,0.026353,0.155041,534.303228,8546
2,"[20, 10, 10, 5, 1]",-1,0.0,0.1,0.1,0.0,0.0,10,2,Adam,2000,True,False,0.01,418697,0.159850,0.025592,0.155678,699.508826,8546
